In [ ]:
import os
import urllib
import pandas as pd
from azure.storage.blob import BlobServiceClient
from sqlalchemy import create_engine
from dotenv import load_dotenv

# تحميل المتغيرات البيئية بأمان من ملف .env المحلي
load_dotenv()

# ==============================================================================
# 1. مرحلة الاستخراج (EXTRACT) - سحب البيانات من السحابة
# ==============================================================================
print("⏳ Connecting to Azure Blob Storage (Bronze Layer)...")

# قراءة البيانات السرية من المتغيرات البيئية لمنع تسريبها على GitHub
BLOB_CONN_STRING = os.getenv("BLOB_CONN_STRING")
CONTAINER_NAME = "bronze-layer"
BLOB_NAME = "ecommerce_data.csv"
LOCAL_FILE_PATH = "temp_ecommerce_data.csv"

if not BLOB_CONN_STRING:
    raise ValueError("⚠️ Error: BLOB_CONN_STRING is missing in environment variables.")

# الاتصال بالمخزن السحابي وتحميل الملف مؤقتاً
blob_service_client = BlobServiceClient.from_connection_string(BLOB_CONN_STRING)
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=BLOB_NAME)

with open(LOCAL_FILE_PATH, "wb") as download_file:
    download_file.write(blob_client.download_blob().readall())

df = pd.read_csv(LOCAL_FILE_PATH)
print(f"✅ Raw Data Loaded Successfully. Shape: {df.shape}")


# ==============================================================================
# 2. مرحلة التنظيف والتحويل (TRANSFORM) - معالجة البيانات وهندستها
# ==============================================================================
print("⏳ Transforming and cleaning data...")

# أ) فلترة البيانات: استبعاد الطلبات الملغية لحماية دقة حسابات الأرباح
df = df[df['order_status'] != 'Cancelled']

# ب) توحيد صيغ التواريخ
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# ج) هندسة الميزات (Feature Engineering): حساب مدة الشحن بالأيام
df['shipping_duration_days'] = (df['ship_date'] - df['order_date']).dt.days

# د) معالجة التقييمات: ترك القيم الفارغة (Nulls) بدون تعويض بأصفار لضمان دقة المتوسط الحسابي
# (تمت صيانة الكود ليتجاهل التقييمات المفقودة بدلاً من حسابها كـ 0)

# هـ) التحقق المالي من صحة البيانات (Data Validation)
df['net_amount_validated'] = df['gross_amount'] - df['discount_amount']

print(f"✅ Transformation Complete. Cleaned Data Shape: {df.shape}")


# ==============================================================================
# 3. مرحلة التحميل (LOAD) - صب البيانات في الـ Cloud Data Warehouse
# ==============================================================================
print("⏳ Connecting to Azure SQL Database (Silver Layer)...")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_DRIVER = '{ODBC Driver 18 for SQL Server}'

# إعداد ماسورة الاتصال الآمن بقاعدة البيانات
connection_params = urllib.parse.quote_plus(
    f'DRIVER={DB_DRIVER};SERVER={DB_SERVER};DATABASE={DB_NAME};UID={DB_USER};PWD={DB_PASS}'
)
db_engine = create_engine(f'mssql+pyodbc:///?odbc_connect={connection_params}')

# إرسال البيانات إلى جدول الطبقة الفضية (Silver Layer)
df.to_sql(name='silver_ecommerce_orders', con=db_engine, if_exists='append', index=False)

# تنظيف الجهاز وحذف الملف المؤقت بعد إتمام الرفع السحابي
if os.path.exists(LOCAL_FILE_PATH):
    os.remove(LOCAL_FILE_PATH)

print("🎯 Success! Pipeline executed completely. Data is live on Azure SQL Database.")